In [2]:
# ============================================================================
# 04_evaluation.ipynb
# Final Model Evaluation & Interpretation
# Dataset: BRFSS 2015 - Diabetes Health Indicators (3 Classes)
#
# FOCUS:
#   - Deep-dive evaluation of the best model (by Recall_Diabetes)
#   - Per-class analysis, error analysis, threshold tuning
#   - Clinical interpretation of results
# ============================================================================

import os
import sys
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

warnings.filterwarnings("ignore")

notebook_dir = os.getcwd()
src_path = os.path.join(notebook_dir, "..", "src", "core")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

# ── feature_engineering.py ──────────────────────────────────────────────────
from feature_engineering import (
    apply_all_feature_engineering,   # full pipeline
    create_health_risk_score,        # individual step (used in Step 8)
    create_lifestyle_score,
    create_interaction_features,
)

# ── modeling.py (src/core) ───────────────────────────────────────────────────
from modeling import (
    load_model,                      # load saved .pkl models
    evaluate_model,                  # returns metrics dict incl. Recall_Diabetes
    compare_models,                  # builds comparison DataFrame
    get_feature_importance,          # extracts feature_importances_ from tree models
    plot_confusion_matrix,           # confusion matrix heatmap
    plot_classification_report,      # per-class precision/recall/f1 heatmap
    plot_roc_curves,                 # OvR ROC curves per class
)

# ── visualization.py ─────────────────────────────────────────────────────────
from visualization import (
    plot_feature_importance,         # horizontal barplot for importance_df
    plot_roc_curves as viz_roc_curves,  # alternative ROC from visualization.py
)

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    precision_recall_curve,
)
from sklearn.preprocessing import StandardScaler, label_binarize

#sns.set_style("whitegrid")
#sns.set_palette("colorblind")
#plt.rcParams["figure.figsize"] = (12, 6)
#plt.rcParams["font.size"] = 10

#output_dir = "../outputs/figures/evaluation"
#os.makedirs(output_dir, exist_ok=True)
#print(f"Output directory: {output_dir}")

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)

output_dir = "../outputs/figures/evaluation"
os.makedirs(output_dir, exist_ok=True)
models_dir = "../outputs/models"



# 1. Load Data

In [3]:
# =============================================================================
# 1. LOAD DATA
# =============================================================================
print("\n" + "=" * 80)
print("STEP 1: LOAD DATA")
print("=" * 80)

features_train = pd.read_csv("../data/processed/features_train_scaled.csv")
features_test  = pd.read_csv("../data/processed/features_test_scaled.csv")
target_train   = pd.read_csv("../data/processed/target_train.csv").squeeze().astype(int) # pyright: ignore[reportAttributeAccessIssue]
target_test    = pd.read_csv("../data/processed/target_test.csv").squeeze().astype(int) # pyright: ignore[reportAttributeAccessIssue]

# ── feature_engineering.apply_all_feature_engineering ───────────────────────
features_train_eng = apply_all_feature_engineering(features_train)
features_test_eng  = apply_all_feature_engineering(features_test)

new_continuous_features = [
    "HealthRiskScore", "LifestyleScore",
    "BMI_x_HighBP", "Age_x_BMI", "GenHlth_x_PhysActivity"
]

scaler = StandardScaler()
features_train_eng[new_continuous_features] = scaler.fit_transform(
    features_train_eng[new_continuous_features]
)
features_test_eng[new_continuous_features] = scaler.transform(
    features_test_eng[new_continuous_features]
)

print(f"Test set: {features_test_eng.shape}")
print(f"Target distribution:\n{target_test.value_counts().sort_index()}")




STEP 1: LOAD DATA
Feature engineering completed. New shape: (183824, 33)
Feature engineering completed. New shape: (45957, 33)
Test set: (45957, 33)
Target distribution:
Diabetes_012
0    38012
1      926
2     7019
Name: count, dtype: int64


# 2. Load Models & Comparison Results

In [5]:
# =============================================================================
# 2. LOAD MODELS & COMPARISON RESULTS
# =============================================================================
print("\n" + "=" * 80)
print("STEP 2: LOAD MODELS & COMPARISON RESULTS")
print("=" * 80)

comparison_df = pd.read_csv(f"{models_dir}/model_comparison_results.csv", index_col=0)

model_files = {
    "Logistic Regression (Baseline)": "Logistic_Regression_Baseline.pkl",
    "Random Forest (Baseline)":       "Random_Forest_Baseline.pkl",
    "XGBoost (Baseline)":             "XGBoost_Baseline.pkl",
    "LinearSVM (10%) (Baseline)":     "LinearSVM_10pct_Baseline.pkl",
    "Logistic Regression (SMOTE)":    "Logistic_Regression_SMOTE.pkl",
    "Random Forest (SMOTE)":          "Random_Forest_SMOTE.pkl",
    "XGBoost (SMOTE)":                "XGBoost_SMOTE.pkl",
    "LinearSVM (10%) (SMOTE)":        "LinearSVM_10pct_SMOTE.pkl",
}

# ── modeling.load_model ──────────────────────────────────────────────────────
all_models = {}
for model_name, filename in model_files.items():
    filepath = os.path.join(models_dir, filename)
    if os.path.exists(filepath):
        all_models[model_name] = load_model(filepath)
    else:
        print(f"Not found: {filepath}")

print(f"\n {len(all_models)} models loaded")



STEP 2: LOAD MODELS & COMPARISON RESULTS
Model loaded: ../outputs/models\Logistic_Regression_Baseline.pkl
Model loaded: ../outputs/models\Random_Forest_Baseline.pkl
Model loaded: ../outputs/models\XGBoost_Baseline.pkl
Model loaded: ../outputs/models\LinearSVM_10pct_Baseline.pkl
Model loaded: ../outputs/models\Logistic_Regression_SMOTE.pkl
Model loaded: ../outputs/models\Random_Forest_SMOTE.pkl
Model loaded: ../outputs/models\XGBoost_SMOTE.pkl
Model loaded: ../outputs/models\LinearSVM_10pct_SMOTE.pkl

 8 models loaded


# 3. Best Model Identification

In [9]:
# =============================================================================
# 3. BEST MODEL IDENTIFICATION
# =============================================================================
print("\n" + "=" * 80)
print("STEP 3: BEST MODEL IDENTIFICATION (PRIMARY: Recall_Diabetes)")
print("=" * 80)

display_cols = ["Accuracy", "Precision", "Recall", "Recall_Diabetes", "F1 (Macro)", "ROC-AUC"]
comparison_sorted = comparison_df[display_cols].sort_values("Recall_Diabetes", ascending=False)

print("\nFULL COMPARISON TABLE (sorted by Recall_Diabetes ↓):")
print("-" * 80)
print(comparison_sorted.round(4).to_string())

best_model_name = comparison_sorted["Recall_Diabetes"].idxmax()
best_model      = all_models.get(best_model_name)

print(f"\nBEST MODEL: {best_model_name}")
print(f"   Recall (Diabetes): {comparison_sorted.loc[best_model_name, 'Recall_Diabetes']:.4f}  <- PRIMARY METRIC")
print(f"   Recall (Weighted): {comparison_sorted.loc[best_model_name, 'Recall']:.4f}")
print(f"   ROC-AUC:           {comparison_sorted.loc[best_model_name, 'ROC-AUC']:.4f}")



STEP 3: BEST MODEL IDENTIFICATION (PRIMARY: Recall_Diabetes)

FULL COMPARISON TABLE (sorted by Recall_Diabetes ↓):
--------------------------------------------------------------------------------
                                Accuracy  Precision  Recall  Recall_Diabetes  F1 (Macro)  ROC-AUC
Logistic Regression (Baseline)    0.6279     0.8382  0.6279           0.5887      0.4264   0.7607
Logistic Regression (SMOTE)       0.6243     0.8371  0.6243           0.5672      0.4251   0.7567
LinearSVM (10%) (SMOTE)           0.6290     0.8347  0.6290           0.5640      0.4244   0.7519
Random Forest (SMOTE)             0.8126     0.7784  0.8126           0.3009      0.4171   0.7141
XGBoost (SMOTE)                   0.8330     0.7983  0.8330           0.2801      0.4239   0.7417
XGBoost (Baseline)                0.8358     0.7901  0.8358           0.1953      0.3996   0.7680
LinearSVM (10%) (Baseline)        0.8354     0.7886  0.8354           0.1731      0.3913   0.7689
Random Forest (Base